Name: Pothapu Lakshmi Pranav Reddy
StudentID:IITP_AIMLT_2601559

# Task 1: Create Database
This section uses sqlite3 to create the database file and random to generate the 40 synthetic records.

In [1]:
import sqlite3
import pandas as pd
import random

# Connect to or create the database
conn = sqlite3.connect('employees.db')
cursor = conn.cursor()

# Create the employees table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS employees (
        emp_id INTEGER PRIMARY KEY,
        name TEXT,
        department TEXT,
        salary REAL,
        years_experience INTEGER,
        performance_score REAL
    )
''')

# Data generation parameters
departments = ['Engineering', 'Sales', 'Marketing', 'HR', 'Finance']
employee_data = []

for i in range(1, 41):
    emp_id = i
    name = f"Employee_{i}"
    department = random.choice(departments)
    salary = round(random.uniform(50000, 150000), 2)
    years_exp = random.randint(1, 15)
    perf_score = round(random.uniform(1.0, 5.0), 1)

    employee_data.append((emp_id, name, department, salary, years_exp, perf_score))

# Insert data into the table
cursor.executemany('''
    INSERT INTO employees (emp_id, name, department, salary, years_experience, performance_score)
    VALUES (?, ?, ?, ?, ?, ?)
''', employee_data)

conn.commit()
print("Database created and 40 records inserted successfully.")

Database created and 40 records inserted successfully.


# Task 2: SQL Queries
Here, we execute the specific queries requested and load them directly into Pandas DataFrames.

In [2]:
# Query 1: High performers with experience
query1 = '''
SELECT name, department, salary, performance_score
FROM employees
WHERE performance_score >= 4.0 AND years_experience >= 3
ORDER BY performance_score DESC
LIMIT 15
'''
df_high_performers = pd.read_sql_query(query1, conn)
print("--- Query 1 Result ---")
print(df_high_performers)

# Query 2: Specific departments and salary range
query2 = '''
SELECT * FROM employees
WHERE (salary BETWEEN 70000 AND 110000)
AND (department = 'Engineering' OR department = 'Sales')
ORDER BY department ASC, salary DESC
'''
df_dept_salary = pd.read_sql_query(query2, conn)
print("\n--- Query 2 Result ---")
print(df_dept_salary)

# Query 3: Departmental Averages
query3 = '''
SELECT department, COUNT(*) as employee_count, AVG(salary) as avg_salary
FROM employees
GROUP BY department
ORDER BY avg_salary DESC
'''
df_avg_salary = pd.read_sql_query(query3, conn)
print("\n--- Query 3 Result ---")
print(df_avg_salary)

--- Query 1 Result ---
          name   department     salary  performance_score
0  Employee_32           HR   77240.94                4.9
1  Employee_14  Engineering   91154.02                4.6
2   Employee_5        Sales  130882.64                4.5
3  Employee_25  Engineering   77319.22                4.5
4  Employee_33  Engineering   57997.47                4.5
5  Employee_18  Engineering   82737.99                4.4
6  Employee_26    Marketing   87765.12                4.0

--- Query 2 Result ---
   emp_id         name   department     salary  years_experience  \
0      14  Employee_14  Engineering   91154.02                 7   
1      18  Employee_18  Engineering   82737.99                 5   
2      25  Employee_25  Engineering   77319.22                 8   
3      23  Employee_23        Sales  108385.93                 7   
4      24  Employee_24        Sales   83465.35                 7   

   performance_score  
0                4.6  
1                4.4  
2          

# Task 3: Pandas Analysis
This section performs the same logic as Task 2, but using native Pandas methods instead of SQL.

In [3]:
# Load the full table for Pandas analysis
df_all = pd.read_sql_query("SELECT * FROM employees", conn)

# 1. Average performance_score by department
avg_perf = df_all.groupby('department')['performance_score'].mean()
print("--- Average Performance by Department ---")
print(avg_perf)

# 2. Replicate Query 1 using Pandas
# Filtering, Sorting, and Selecting specific columns
df_replicated = df_all.loc[
    (df_all['performance_score'] >= 4.0) & (df_all['years_experience'] >= 3),
    ['name', 'department', 'salary', 'performance_score']
].sort_values(by='performance_score', ascending=False).head(15)

print("\n--- Replicated Query 1 (Pandas) ---")
print(df_replicated)

# Close connection
conn.close()

--- Average Performance by Department ---
department
Engineering    3.400000
Finance        2.300000
HR             2.850000
Marketing      2.566667
Sales          2.571429
Name: performance_score, dtype: float64

--- Replicated Query 1 (Pandas) ---
           name   department     salary  performance_score
31  Employee_32           HR   77240.94                4.9
13  Employee_14  Engineering   91154.02                4.6
4    Employee_5        Sales  130882.64                4.5
32  Employee_33  Engineering   57997.47                4.5
24  Employee_25  Engineering   77319.22                4.5
17  Employee_18  Engineering   82737.99                4.4
25  Employee_26    Marketing   87765.12                4.0
